In [17]:
# 00_data_filtering_rebuild

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

In [18]:
# =========================
# Config
# =========================
RAW_DATA_DIR = Path("../../data")
SAVE_ROOT = Path("../../DifferentCom_data_rebuild")



META_PATHS = {
    "met": RAW_DATA_DIR / "human_metabolomics_meta.txt",
    "prot": RAW_DATA_DIR / "human_proteomics_meta.txt",
    "rna": RAW_DATA_DIR / "human_miRNA_meta.txt",
}

MATRIX_PATHS = {
    "met": RAW_DATA_DIR / "human_metabolomics.txt",
    "prot": RAW_DATA_DIR / "human_proteomics.txt",
    "rna": RAW_DATA_DIR / "human_miRNA.txt",
}

COMBO_MAP = {
    "A": ["prot"],
    "B": ["met"],
    "C": ["rna"],
    "AB": ["prot", "met"],
    "AC": ["prot", "rna"],
    "BC": ["met", "rna"],
    "ABC": ["prot", "met", "rna"],
}

COMBOS= [
    "A",
    "B",
    "C",
    "AB",
    "AC",
    "BC",
    "ABC"
]

OMICS_PREFIX = {
    "prot": "PROT",
    "met": "MET",
    "rna": "RNA",
}

TISSUES = ["CSF", "Serum"]
TIMEPOINTS = [24, 48, 72, 96, 120]
TP_DIR = Path("../../DifferentCom_data_rebuild/tp_views")


SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("SAVE_ROOT:", SAVE_ROOT.resolve())

SAVE_ROOT: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild


In [19]:
def replace_floor_with_nan(
    df: pd.DataFrame,
    id_col: str = "Gene",
    floor_quantile: float = 0.001,
) -> pd.DataFrame:
    """
    Numeric part에서 극단적으로 낮은 floor value를 NaN으로 치환.
    단일 고정 바닥값이 반복되는 경우를 우선 제거하기 위한 보수적 처리.
    """
    out = df.copy()

    value_cols = [c for c in out.columns if c != id_col]
    out[value_cols] = out[value_cols].apply(pd.to_numeric, errors="coerce")

    vals = out[value_cols].to_numpy().ravel()
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return out

    floor_val = np.quantile(vals, floor_quantile)

    # floor_val 이하 값들을 NaN 처리
    out[value_cols] = out[value_cols].mask(out[value_cols] <= floor_val, np.nan)
    return out

In [20]:
# =========================
# Raw loaders
# =========================
def read_table(path: Path, sep: str = "\t") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, sep=sep)

met_raw = replace_floor_with_nan(read_table(MATRIX_PATHS["met"]), id_col="Gene")
prot_raw = replace_floor_with_nan(read_table(MATRIX_PATHS["prot"]), id_col="Gene")
rna_raw = replace_floor_with_nan(read_table(MATRIX_PATHS["rna"]), id_col="Gene")

met_meta = read_table(META_PATHS["met"])
prot_meta = read_table(META_PATHS["prot"])
rna_meta = read_table(META_PATHS["rna"])

print("met_raw :", met_raw.shape)
print("prot_raw:", prot_raw.shape)
print("rna_raw :", rna_raw.shape)
print("met_meta:", met_meta.shape)
print("prot_meta:", prot_meta.shape)
print("rna_meta:", rna_meta.shape)

met_raw : (749, 806)
prot_raw: (438, 929)
rna_raw : (312, 375)
met_meta: (758, 24)
prot_meta: (842, 28)
rna_meta: (366, 27)


In [21]:
# =========================
# Helpers
# =========================
def _find_feature_id_col(df: pd.DataFrame) -> str:
    candidates = [
        "Gene", "gene", "Protein", "protein", "Metabolite", "metabolite",
        "Feature", "feature", "ID", "id", "Name", "name"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return df.columns[0]


def integrate_data(meta: pd.DataFrame, matrix_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
        X_sample: sample x feature matrix
        meta_aligned: metadata aligned to X_sample rows
    """
    meta = meta.copy()
    matrix_df = matrix_df.copy()

    if "Sample" not in meta.columns:
        raise ValueError("'Sample' column is required in metadata")
    if "Patient" not in meta.columns:
        raise ValueError("'Patient' column is required in metadata")
    if "Timepoint" not in meta.columns:
        raise ValueError("'Timepoint' column is required in metadata")

    feature_id_col = _find_feature_id_col(matrix_df)

    X_sample = matrix_df.set_index(feature_id_col).T
    X_sample.index = X_sample.index.astype(str).str.strip()
    X_sample.index.name = "Sample"

    meta["Sample"] = meta["Sample"].astype(str).str.strip()

    common = X_sample.index.intersection(meta["Sample"])
    X_sample = X_sample.loc[common].copy()
    meta_aligned = (
        meta.set_index("Sample")
            .loc[common]
            .reset_index()
    )

    return X_sample, meta_aligned

In [ ]:
def select_clinical_columns(X: pd.DataFrame) -> pd.DataFrame:
    clinical_cols = [
        "Age", "Gender", "Level",
    ]
    keep_cols = [c for c in clinical_cols if c in X.columns]
    return X.loc[:, keep_cols].copy()

In [23]:
def build_patient_time_features(
    X_sample: pd.DataFrame,
    meta_aligned: pd.DataFrame,
    tissue: str,
    agg: str = "mean",
) -> pd.DataFrame:
    meta_use = meta_aligned.copy()
    if "Tissue" in meta_use.columns:
        meta_use = meta_use[meta_use["Tissue"].astype(str).str.lower() == tissue.lower()].copy()

    sample_ids = meta_use["Sample"].astype(str).tolist()
    X_use = X_sample.loc[sample_ids].copy()

    meta_use = meta_use.set_index("Sample").loc[X_use.index].reset_index()

    blocks = []
    for tp in TIMEPOINTS:
        idx = meta_use["Timepoint"] == tp
        if idx.sum() == 0:
            continue

        X_tp = X_use.loc[idx.values].copy()
        pt_tp = meta_use.loc[idx, "Patient"].astype(str).values

        X_tp = X_tp.copy()
        X_tp["Patient"] = pt_tp

        if agg == "mean":
            X_pt = X_tp.groupby("Patient").mean(numeric_only=True)
        else:
            raise ValueError("Currently only agg='mean' is supported")

        X_pt.columns = [f"{c}_TP{tp}" for c in X_pt.columns]
        blocks.append(X_pt)

    if not blocks:
        return pd.DataFrame()

    out = pd.concat(blocks, axis=1)
    out.index = out.index.astype(str)
    return out

In [24]:
def add_adjacent_delta_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    tp_cols = [c for c in df.columns if "_TP" in c]
    bases = sorted({c.split("_TP")[0] for c in tp_cols})

    new_cols = {}
    for base in bases:
        for tp_prev, tp_curr in zip(TIMEPOINTS[:-1], TIMEPOINTS[1:]):
            c_prev = f"{base}_TP{tp_prev}"
            c_curr = f"{base}_TP{tp_curr}"
            if c_prev in df.columns and c_curr in df.columns:
                new_cols[f"{base}_d{tp_prev}_{tp_curr}"] = df[c_curr] - df[c_prev]

    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df

In [25]:
def build_target_by_patient(meta_df: pd.DataFrame, target_col: str = "DeltaTMS") -> pd.Series:
    if target_col not in meta_df.columns:
        raise ValueError(f"'{target_col}' not found in metadata")

    y = (
        meta_df[["Patient", target_col]]
        .dropna(subset=[target_col])
        .drop_duplicates(subset=["Patient"])
        .copy()
    )
    y["Patient"] = y["Patient"].astype(str)
    return y.set_index("Patient")[target_col].astype(float)


def align_on_patients(
    X: pd.DataFrame,
    y: pd.Series | None = None,
    clinical: pd.DataFrame | None = None,
):
    idx = X.index.astype(str)
    if y is not None:
        idx = idx.intersection(y.index.astype(str))
    if clinical is not None:
        idx = idx.intersection(clinical.index.astype(str))

    X2 = X.loc[idx].copy()
    y2 = y.loc[idx].copy() if y is not None else None
    c2 = clinical.loc[idx].copy() if clinical is not None else None
    return X2, y2, c2


In [26]:
def select_tp_omics_only_columns(X: pd.DataFrame, tp: int) -> pd.DataFrame:
    tp_tag = f"_TP{tp}"
    cols_tp = [c for c in X.columns if c.endswith(tp_tag)]
    return X.loc[:, cols_tp].copy()


def select_tp_omics_plus_delta_columns(X: pd.DataFrame, tp: int) -> pd.DataFrame:
    X_delta = add_adjacent_delta_features(X)

    current_tp_cols = [c for c in X_delta.columns if c.endswith(f"_TP{tp}")]

    delta_cols = []
    for tp_prev, tp_curr in zip(TIMEPOINTS[:-1], TIMEPOINTS[1:]):
        if tp_curr <= tp:
            tag = f"_d{tp_prev}_{tp_curr}"
            delta_cols.extend([c for c in X_delta.columns if c.endswith(tag)])

    selected_cols = list(dict.fromkeys(current_tp_cols + delta_cols))
    return X_delta.loc[:, selected_cols].copy()

In [ ]:
# =========================
# Fold-safe unsupervised filtering helpers
# =========================
DEFAULT_MISS_THRESH = {
    "prot": 0.4,
    "met": 0.4,
    "rna": 0.4,
    "combo": 0.4,
}



def drop_useless_cols(
    X: pd.DataFrame,
    miss_thresh: float,
    keep_cols: Iterable[str] | None = None,
    return_info: bool = False,
):
    """
    [전]
    Lasso notebook의 filtering 아이디어를 그대로 전체 데이터에 먼저 적용하면
    CV 바깥에서 keep columns가 결정되어 leakage 위험이 생길 수 있습니다.

    [후]
    이 함수는 '현재 입력된 X'에 대해서만
    1) constant columns 제거
    2) high-missing columns 제거
    를 수행합니다.

    따라서 later CV notebook에서는 반드시:
        outer-train subset -> fit_fold_safe_filter()
        valid/test          -> apply_fold_safe_filter()
    순서로 사용해야 합니다.
    """
    if keep_cols is None:
        keep_cols = []

    keep_cols = [c for c in keep_cols if c in X.columns]

    # [수정 포인트 1]
    # 상수 컬럼 제거 (단, 보호 clinical columns은 제외)
    nunique = X.nunique(dropna=True)
    const_cols = [
        c for c in nunique.index
        if nunique[c] <= 1 and c not in keep_cols
    ]
    X1 = X.drop(columns=const_cols)

    # [수정 포인트 2]
    # missing 비율이 threshold 이상인 컬럼 제거
    miss = X1.isna().mean()
    high_miss_cols = [
        c for c in miss.index
        if miss[c] >= miss_thresh and c not in keep_cols
    ]
    X2 = X1.drop(columns=high_miss_cols)

    info = {
        "input_n_cols": int(X.shape[1]),
        "kept_n_cols": int(X2.shape[1]),
        "const_cols": const_cols,
        "high_miss_cols": high_miss_cols,
        "keep_cols": keep_cols,
        "miss_thresh": miss_thresh,
    }

    if return_info:
        return X2, info
    return X2



def fit_fold_safe_filter(
    X_train: pd.DataFrame,
    miss_thresh: float,
    keep_cols: Iterable[str] | None = None,
) -> dict:
    """
    [핵심 수정]
    전체 데이터가 아니라 outer-train subset에서만
    살아남을 컬럼을 결정합니다.
    """
    X_train_f, info = drop_useless_cols(
        X_train,
        miss_thresh=miss_thresh,
        keep_cols=keep_cols,
        return_info=True,
    )

    return {
        "keep_cols_final": list(X_train_f.columns),
        "const_cols": info["const_cols"],
        "high_miss_cols": info["high_miss_cols"],
        "miss_thresh": miss_thresh,
        "protected_keep_cols": info["keep_cols"],
    }



def apply_fold_safe_filter(X: pd.DataFrame, fitted_filter: dict) -> pd.DataFrame:
    """
    [핵심 수정]
    train subset에서 살아남은 컬럼만 validation/test에 동일 적용합니다.
    """
    keep_cols_final = [c for c in fitted_filter["keep_cols_final"] if c in X.columns]
    return X.loc[:, keep_cols_final].copy()


In [28]:
# =========================
# Build aligned sample-level matrices
# =========================
X_met_sample, meta_met_aligned = integrate_data(met_meta, met_raw)
X_prot_sample, meta_prot_aligned = integrate_data(prot_meta, prot_raw)
X_rna_sample, meta_rna_aligned = integrate_data(rna_meta, rna_raw)

print("X_met_sample :", X_met_sample.shape)
print("X_prot_sample:", X_prot_sample.shape)
print("X_rna_sample :", X_rna_sample.shape)

X_met_sample : (758, 749)
X_prot_sample: (842, 438)
X_rna_sample : (366, 312)


In [29]:
# =========================
# Build patient-level blocks
# =========================
patient_blocks = {}

for omics_name, X_sample, meta_aligned in [
    ("prot", X_prot_sample, meta_prot_aligned),
    ("met", X_met_sample, meta_met_aligned),
    ("rna", X_rna_sample, meta_rna_aligned),
]:
    for tissue in TISSUES:
        X_pt = build_patient_time_features(
            X_sample,
            meta_aligned,
            tissue=tissue,
            agg="mean",
        )
        # Experiment B: delta feature를 생성하지 않음
        patient_blocks[(omics_name, tissue)] = X_pt
        print(f"{omics_name}-{tissue}: {X_pt.shape}")

prot-CSF: (103, 2190)
prot-Serum: (108, 2190)
met-CSF: (83, 3745)
met-Serum: (108, 3745)
rna-CSF: (39, 1560)
rna-Serum: (38, 1560)


In [ ]:
# =========================
# Build clinical / target
# =========================

def build_clinical_by_tissue(meta_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    clinical_cols = ["Age", "Gender", "Level"]

    required_cols = ["Sample", "Patient", "Timepoint", "Tissue"]
    use_cols = [c for c in required_cols + clinical_cols if c in meta_df.columns]

    meta_use = meta_df.loc[:, use_cols].copy()
    meta_use["Patient"] = meta_use["Patient"].astype(str)

    out = {}
    for tissue in TISSUES:
        m = meta_use[meta_use["Tissue"].astype(str).str.lower() == tissue.lower()].copy()

        keep_cols = [c for c in clinical_cols if c in m.columns]
        clinical_pt = (
            m[["Patient"] + keep_cols]
            .drop_duplicates(subset=["Patient"])
            .set_index("Patient")
            .sort_index()
        )
        clinical_pt.index = clinical_pt.index.astype(str)
        out[tissue] = clinical_pt

    return out


y_patient = build_target_by_patient(meta_prot_aligned, target_col="DeltaTMS")
clinical_by_tissue = build_clinical_by_tissue(meta_prot_aligned)

for tissue in TISSUES:
    print(tissue, "clinical:", clinical_by_tissue[tissue].shape)
print("y_patient:", y_patient.shape)

CSF clinical: (103, 7)
Serum clinical: (108, 7)
y_patient: (110,)


In [ ]:
# =========================
# Export patient-level base tables
# =========================
base_dir = SAVE_ROOT / "base_patient_tables"
base_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "timepoints": TIMEPOINTS,
    "tissues": TISSUES,
    "combo_map": COMBO_MAP,
    "notes": [
        "No supervised feature selection in this notebook.",
        "No CV score is computed here.",
        "Clinical block includes only: Age, Gender, Level.",
        "drop_useless_cols() is included for fold-safe filtering usage.",
        "Fold-wise filtering must be fit only on the outer-train subset, then applied to validation/test with saved keep columns.",
    ],
}

y_patient.rename("DeltaTMS").to_csv(base_dir / "target_by_patient.csv", index=True)

for tissue, clinical_df in clinical_by_tissue.items():
    clinical_df.to_csv(base_dir / f"clinical_{tissue.lower()}.csv", index=True)

for (omics_name, tissue), df in patient_blocks.items():
    out = base_dir / f"{omics_name}_{tissue.lower()}_patient_table.csv"
    df.to_csv(out, index=True)

with open(base_dir / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved base tables to:", base_dir)

Saved base tables to: ../../DifferentCom_data_rebuild/base_patient_tables


In [32]:
def build_combo_table(
    combo_key: str,
    tissue: str,
    patient_blocks: dict[tuple[str, str], pd.DataFrame],
    y_patient: pd.Series,
    clinical_by_tissue: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    parts = []

    for omics_name in COMBO_MAP[combo_key]:
        prefix = OMICS_PREFIX[omics_name]
        df = patient_blocks[(omics_name, tissue)].add_prefix(f"{tissue.upper()}_{prefix}_")
        parts.append(df)

    X = pd.concat(parts, axis=1)

    clinical_df = clinical_by_tissue[tissue].copy()

    # target + clinical + omics 공통 patient만 남기고
    # clinical을 실제로 붙인다
    X, _, clinical_df = align_on_patients(X, y=y_patient, clinical=clinical_df)
    X = pd.concat([clinical_df, X], axis=1)

    return X

In [33]:
# ==========================================
# Export clinical-only tp views
# ==========================================
tp_dir = SAVE_ROOT / "tp_views"
tp_dir.mkdir(parents=True, exist_ok=True)

combo_dir = SAVE_ROOT / "combo_patient_tables"
combo_dir.mkdir(parents=True, exist_ok=True)

for combo_key in COMBO_MAP:
    for tissue in TISSUES:
        X_combo = build_combo_table(
            combo_key=combo_key,
            tissue=tissue,
            patient_blocks=patient_blocks,
            y_patient=y_patient,
            clinical_by_tissue=clinical_by_tissue,
        )

        tissue_short = "csf" if tissue == "CSF" else "ser"
        X_combo.to_csv(combo_dir / f"{tissue_short}_{combo_key}.csv", index=True)
        print(f"saved: {tissue_short}_{combo_key}.csv {X_combo.shape}")

saved: csf_A.csv (103, 2197)
saved: ser_A.csv (108, 2197)
saved: csf_B.csv (83, 3752)
saved: ser_B.csv (108, 3752)
saved: csf_C.csv (39, 1567)
saved: ser_C.csv (38, 1567)
saved: csf_AB.csv (103, 5942)
saved: ser_AB.csv (108, 5942)
saved: csf_AC.csv (103, 3757)
saved: ser_AC.csv (108, 3757)
saved: csf_BC.csv (83, 5312)
saved: ser_BC.csv (108, 5312)
saved: csf_ABC.csv (103, 7502)
saved: ser_ABC.csv (108, 7502)


In [34]:
# =========================
# Build TP-specific views
# =========================
tp_dir = SAVE_ROOT / "tp_views"
tp_dir.mkdir(parents=True, exist_ok=True)

for combo_key in COMBO_MAP:
    for tissue in TISSUES:
        tissue_short = "csf" if tissue == "CSF" else "ser"

        combo_path = combo_dir / f"{tissue_short}_{combo_key}.csv"
        X_combo = pd.read_csv(combo_path, index_col=0)
        X_combo.index = X_combo.index.astype(str)

        for tp in TIMEPOINTS:
            # 1) clinical-only
            X_clinical = select_clinical_columns(X_combo)
            X_clinical.to_csv(
                tp_dir / f"x_clinical_{tissue_short}_{combo_key}_{tp}.csv",
                index=True,
                index_label="Patient",
            )
            print(f"saved: x_clinical_{tissue_short}_{combo_key}_{tp}.csv {X_clinical.shape}")

            # 2) omics-only
            X_omics = select_tp_omics_only_columns(X_combo, tp=tp)
            X_omics.to_csv(
                tp_dir / f"x_omics_{tissue_short}_{combo_key}_{tp}.csv",
                index=True,
                index_label="Patient",
            )
            print(f"saved: x_omics_{tissue_short}_{combo_key}_{tp}.csv {X_omics.shape}")

            # 3) omics + delta
            X_omicsdelta = select_tp_omics_plus_delta_columns(X_combo, tp=tp)
            X_omicsdelta.to_csv(
                tp_dir / f"x_omicsdelta_{tissue_short}_{combo_key}_{tp}.csv",
                index=True,
                index_label="Patient",
            )
            print(f"saved: x_omicsdelta_{tissue_short}_{combo_key}_{tp}.csv {X_omicsdelta.shape}")

print("Saved TP-specific clinical/omics views to:", tp_dir)

saved: x_clinical_csf_A_24.csv (103, 7)
saved: x_omics_csf_A_24.csv (103, 438)
saved: x_omicsdelta_csf_A_24.csv (103, 438)
saved: x_clinical_csf_A_48.csv (103, 7)
saved: x_omics_csf_A_48.csv (103, 438)
saved: x_omicsdelta_csf_A_48.csv (103, 874)
saved: x_clinical_csf_A_72.csv (103, 7)
saved: x_omics_csf_A_72.csv (103, 438)
saved: x_omicsdelta_csf_A_72.csv (103, 1310)
saved: x_clinical_csf_A_96.csv (103, 7)
saved: x_omics_csf_A_96.csv (103, 438)
saved: x_omicsdelta_csf_A_96.csv (103, 1746)
saved: x_clinical_csf_A_120.csv (103, 7)
saved: x_omics_csf_A_120.csv (103, 438)
saved: x_omicsdelta_csf_A_120.csv (103, 2182)
saved: x_clinical_ser_A_24.csv (108, 7)
saved: x_omics_ser_A_24.csv (108, 438)
saved: x_omicsdelta_ser_A_24.csv (108, 438)
saved: x_clinical_ser_A_48.csv (108, 7)
saved: x_omics_ser_A_48.csv (108, 438)
saved: x_omicsdelta_ser_A_48.csv (108, 874)
saved: x_clinical_ser_A_72.csv (108, 7)
saved: x_omics_ser_A_72.csv (108, 438)
saved: x_omicsdelta_ser_A_72.csv (108, 1310)
saved: x_

In [36]:
# =========================
# Reference: leakage-safe filtering usage pattern
# =========================
example_x = pd.read_csv(combo_dir / "csf_ABC.csv", index_col=0)
example_x.index = example_x.index.astype(str)

example_train_ids = example_x.index[: max(1, int(len(example_x) * 0.7))]
example_valid_ids = example_x.index.difference(example_train_ids)

X_train_example = example_x.loc[example_train_ids].copy()
X_valid_example = example_x.loc[example_valid_ids].copy()

# [전]
# example_x 전체에서 filtering 기준을 먼저 정하면 leakage 위험이 있습니다.
#
# [후]
# train subset에서만 keep columns를 결정합니다.
example_keep_cols = [
    c for c in [
        "Age", "Gender", "Level", "AIS",
        "AIS_Base_Numeric", "TMS_Base", "UEMS_Base", "LEMS_Base"
    ]
    if c in X_train_example.columns
]

fitted_filter = fit_fold_safe_filter(
    X_train_example,
    miss_thresh=DEFAULT_MISS_THRESH["combo"],
    keep_cols=example_keep_cols,
)

# train에서 결정된 컬럼만 valid에 적용
X_train_filtered = apply_fold_safe_filter(X_train_example, fitted_filter)
X_valid_filtered = apply_fold_safe_filter(X_valid_example, fitted_filter)

print("example_x:", example_x.shape)
print("X_train_filtered:", X_train_filtered.shape)
print("X_valid_filtered:", X_valid_filtered.shape)
print("protected clinical cols:", fitted_filter["protected_keep_cols"])
print("dropped const:", len(fitted_filter["const_cols"]))
print("dropped high-missing:", len(fitted_filter["high_miss_cols"]))


example_x: (103, 7502)
X_train_filtered: (72, 2352)
X_valid_filtered: (31, 2352)
protected clinical cols: ['Age', 'Gender', 'Level', 'AIS_Base_Numeric', 'TMS_Base', 'UEMS_Base', 'LEMS_Base']
dropped const: 2070
dropped high-missing: 3080


In [37]:
# =========================
# Quick sanity check
# =========================
example_files = [
    base_dir / "prot_csf_patient_table.csv",
    combo_dir / "csf_A.csv",
    base_dir / "target_by_patient.csv",
]

for p in example_files:
    df = pd.read_csv(p, index_col=0)
    print(p.name, df.shape)
    display(df.head(2))

prot_csf_patient_table.csv (103, 2190)


,A1BG|GVTFLLR_TP24,A1BG|LETPDFQLFK_TP24,A2M|AIGYLNTGYQR_TP24,ACO1|NIEVPFKPAR_TP24,ACTA2|SYELPDGQVITIGNER_TP24,ACTN4|LSGSNPYTTVTPQIINSK_TP24,ACY1|LHEAVFLR_TP24,ACYP2|LEYSNFSIR_TP24,ADAMTS1|GAFYLLGEAYFIQPLPAASER_TP24,ADD1|VDENNPEYLR_TP24,...,VWF|HIVTFDGQNFK_TP120,VWF|ILAGPAGDSNVVK_TP120,WARS|HVTFNQVK_TP120,WDR1|YEYQPFAGK_TP120,WFIKKN2|ADFPLSVVR_TP120,YWHAB|YLSEVASGDNK_TP120,YWHAE|EAAENSLVAYK_TP120,YWHAG|YLAEVATGEK_TP120,YWHAH|LAEQAER_TP120,YWHAQ|AVTEQGAELSNEER_TP120
Patient,,,,,,,,,,,,,,,,,,,,,
D02,1.610278,NaN,NaN,NaN,NaN,NaN,-3.835062,-2.080242,-3.324236,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
D06,1.687583,NaN,NaN,NaN,NaN,NaN,-3.868006,NaN,-3.634391,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


csf_A.csv (103, 2197)


,Age,Gender,Level,AIS_Base_Numeric,TMS_Base,UEMS_Base,LEMS_Base,CSF_PROT_A1BG|GVTFLLR_TP24,CSF_PROT_A1BG|LETPDFQLFK_TP24,CSF_PROT_A2M|AIGYLNTGYQR_TP24,...,CSF_PROT_VWF|HIVTFDGQNFK_TP120,CSF_PROT_VWF|ILAGPAGDSNVVK_TP120,CSF_PROT_WARS|HVTFNQVK_TP120,CSF_PROT_WDR1|YEYQPFAGK_TP120,CSF_PROT_WFIKKN2|ADFPLSVVR_TP120,CSF_PROT_YWHAB|YLSEVASGDNK_TP120,CSF_PROT_YWHAE|EAAENSLVAYK_TP120,CSF_PROT_YWHAG|YLAEVATGEK_TP120,CSF_PROT_YWHAH|LAEQAER_TP120,CSF_PROT_YWHAQ|AVTEQGAELSNEER_TP120
Patient,,,,,,,,,,,,,,,,,,,,,
D02,34,M,C04,2,20,20,0,1.610278,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
D06,66,M,C04,1,20,5,15,1.687583,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


target_by_patient.csv (110, 1)


,DeltaTMS
Patient,
D02,9.0
D03,0.0


In [38]:
import re
import pandas as pd

x24 = pd.read_csv(tp_dir / "x_omics_csf_A_24.csv", index_col=0)
x48 = pd.read_csv(tp_dir / "x_omics_csf_A_48.csv", index_col=0)

def strip_tp(col):
    return re.sub(r"_TP\d+$", "", col)

base24 = [strip_tp(c) for c in x24.columns]
base48 = [strip_tp(c) for c in x48.columns]

print("same base feature set:", set(base24) == set(base48))
print("base overlap count   :", len(set(base24) & set(base48)))
print("base24 count         :", len(set(base24)))
print("base48 count         :", len(set(base48)))

same base feature set: True
base overlap count   : 438
base24 count         : 438
base48 count         : 438


In [39]:
x24_norm = x24.copy()
x48_norm = x48.copy()

x24_norm.columns = [strip_tp(c) for c in x24.columns]
x48_norm.columns = [strip_tp(c) for c in x48.columns]

common = sorted(set(x24_norm.columns) & set(x48_norm.columns))

x24c = x24_norm[common].sort_index(axis=1)
x48c = x48_norm[common].sort_index(axis=1)

print("common feature count:", len(common))
print("same values after TP strip:", (x24c.fillna(0) == x48c.fillna(0)).all().all())
print("missing 24:", x24c.isna().mean().mean())
print("missing 48:", x48c.isna().mean().mean())

common feature count: 438
same values after TP strip: False
missing 24: 0.6178348184599014
missing 48: 0.6123597996187438


In [ ]:
# ==========================================
# Quick check: exported tp_views should not contain delta columns
# ==========================================
tissues = ["csf", "ser"]
combos = ["A", "B", "C", "AB", "AC", "BC", "ABC"]
feature_modes = ["omics", "omicsdelta"]

for feature_mode in feature_modes:
    print(f"\n[CHECK] feature_mode={feature_mode}")
    for tissue in tissues:
        for combo in combos:
            for tp in TIMEPOINTS:
                path = TP_DIR / f"x_{feature_mode}_{tissue}_{combo}_{tp}.csv"
                if not path.exists():
                    print("missing:", path.name)
                    continue

                tmp = pd.read_csv(path, nrows=3)

                delta_cols = [c for c in tmp.columns if "_d" in str(c)]
                baseline_cols = [
                    c for c in tmp.columns
                    if c in ["Age", "Gender", "Level"]
                ]

                total_cols = len(tmp.columns) - 1 if "Patient" in tmp.columns else len(tmp.columns)

                print(
                    f"{path.name}: total_cols={total_cols}, "
                    f"delta_cols={len(delta_cols)}, baseline_cols={len(baseline_cols)}"
                )

                if feature_mode == "omics" and len(delta_cols) > 0:
                    print("  [WARN] omics file should not have delta columns:", delta_cols[:10])

                if len(baseline_cols) > 0:
                    print("  [WARN] baseline columns found:", baseline_cols[:10])
                print("  delta examples:", delta_cols[:10])


[CHECK] feature_mode=omics
x_omics_csf_A_24.csv: total_cols=438, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_A_48.csv: total_cols=438, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_A_72.csv: total_cols=438, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_A_96.csv: total_cols=438, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_A_120.csv: total_cols=438, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_B_24.csv: total_cols=749, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_B_48.csv: total_cols=749, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_B_72.csv: total_cols=749, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_B_96.csv: total_cols=749, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_B_120.csv: total_cols=749, delta_cols=0, baseline_cols=0
  delta examples: []
x_omics_csf_C_24.csv: total_cols=312, delta_cols=0, baseline_cols=0
  delta exam